# Part 4: do frontier models resist, or do they know the answers?
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/advaith334/nst2046-memory-sycophancy/blob/main/notebooks/05_frontier.ipynb)

Claude never moved in the main study, but it also got every item right, so a memory had nothing to move. Aditya proposed that frontier models might be "benchmaxxed": robust only on prompts that look like tests. Predictions were committed before any memory result was read. Run 6 November 2026, exploratory.

| Test | Question |
|---|---|
| F1 Headroom | On 64 computed questions, do frontier models ever get things wrong, and does memory move them then? |
| F2 Test awareness | Does a cue that the chat is a test, or a real user, change the effect? |
| F3 Free text | Writing tasks and drafts containing a myth, with and without the myth stored |
| F4 Pressure | An "expert" memory, a repeated memory, and a user who pushes back |

In [ ]:
# Setup. In Colab this downloads the repo. On your own machine, start Jupyter from the repo folder.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("memsyco_lite"):
    !git clone -q https://github.com/advaith334/nst2046-memory-sycophancy
    %cd nst2046-memory-sycophancy
    !pip install --quiet openai
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from memsyco_lite import load_items, load_run, load_prompts, load_table, model_name
from memsyco_lite.data import show_prompt
from memsyco_lite.stats import effect, fmt, rate
pd.set_option("display.max_colwidth", 140)
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

## F1: frontier models almost never got it wrong
Accuracy with no memory on the 64 computed questions.

In [ ]:
f1 = pd.concat([load_run("5-frontier", "f1-frontier"), load_run("5-frontier", "f1-main-models")])
t = f1.groupby("model").agg(correct=("score", lambda s: (s == "correct").sum()), answered=("score", lambda s: s.isin(["correct", "false_option", "other_wrong"]).sum()))
t.index = [model_name(m) for m in t.index]
t.sort_values("correct", ascending=False)

Item-level results for every F test, as reported:

In [ ]:
load_table("5_frontier_item_level")

## F3: a stored myth made models correct it more often
Share of replies the judge scored as openly correcting the myth (%).

In [ ]:
f3 = pd.concat([load_run("5-frontier", "f3-frontier"), load_run("5-frontier", "f3-main-models")])
t = (f3.assign(corrects=f3.outcome.eq("corrects_openly")).groupby(["model", "condition"]).corrects.mean().mul(100).round(0).unstack())
t.index = [model_name(m) for m in t.index]
t

In [ ]:
myths = {i["id"]: i for i in json.load(open("data/items/frontier_myths.json"))}
ex = f3[(f3.model == "claude-sonnet") & (f3.item == "fa_antibiotics_cold") & f3.condition.isin(["generate_none", "generate_stored"])]
print("Task:", myths["fa_antibiotics_cold"]["generate_task"], "\nStored myth:", myths["fa_antibiotics_cold"]["false_claim"], "\n")
for r in ex.itertuples():
    print(f"--- {r.condition}: {r.outcome}\n{r.response[:600]}\n")

A second judge (Claude Haiku) re-read the F3 replies. Agreement with the first judge (DeepSeek V4 Flash). This quick recomputation can differ slightly from the report's 0.76, which came from the project's own judge-check script.

In [ ]:
j2 = load_table("5_f3_second_judge").merge(f3[["trial_id", "outcome"]], on="trial_id").dropna()
agree = (j2.outcome == j2.second_outcome).mean()
cats = sorted(set(j2.outcome) | set(j2.second_outcome))
pe = sum((j2.outcome == c).mean() * (j2.second_outcome == c).mean() for c in cats)
print(f"{len(j2)} replies, agreement {agree:.0%}, Cohen's kappa {(agree - pe) / (1 - pe):.2f}")

## What this shows
- Frontier models were at the ceiling: Sonnet answered 190 of 191 with no memory. Only 14 items left any frontier model unsure, so a null here is weak.
- We found no support for the benchmaxxed account, but we cannot say whether frontier models resist memory or simply know the answer.
- In free writing, a stored myth raised open corrections in every model (Sonnet 25% to 85%), so memory can prompt correction.

**Limits.** Few items, AI judges only, and 165 trials of one F4 track are missing because credit ran out.

## Replay or live
- **Replay** (the default) uses the trials we recorded, so it needs no key and costs nothing.
- **Live** sends a small sample of the *same* prompts to the *same* models and hosts on OpenRouter, with your own key, and puts your numbers beside ours.

Put your key in Colab's 🔑 **Secrets** panel as `OPENROUTER_API_KEY`, or paste it when asked. Models answer at temperature 0.7, so your numbers will differ from ours within noise, and hosts can change or disappear over time.

In [ ]:
MODE = "replay"   # "replay": our recorded results, no key and no cost. "live": rerun a small sample with your own key.
CAP_USD = 0.50    # a live run stops once it has spent this much (US$)

The frontier runs did not keep message logs, so live mode rebuilds the prompts with the study's template (`memsyco_lite.prompts`) on the 14 items that left a model unsure.

In [ ]:
if MODE == "live":
    from memsyco_lite.live import Client, get_key
    from memsyco_lite.parse import score_mcq
    from memsyco_lite.prompts import build_messages, option_order
    head = load_items("frontier_headroom")
    chosen = [i for i in json.load(open("data/items/frontier_headroom.json")) if i["id"] in set(load_run("5-frontier", "f2f4-frontier-base").item)][:7]
    client = Client(get_key(), cap_usd=CAP_USD)
    out = []
    for it in chosen:
        order = option_order(it, seed=it["id"])
        for cond in ["no_belief", "stored_incorrect"]:
            r = client.chat("claude-haiku", build_messages(it, cond, order), max_tokens=8000)
            out.append({"item": it["id"], "condition": cond, "score": score_mcq(r["text"], order, it)[1]})
    display(pd.DataFrame(out).pivot(index="item", columns="condition", values="score"))
    print(f"Spent US${client.spent:.4f}")
else:
    print("Replay mode. Set MODE = 'live' above to try Claude Haiku on 7 of the hard items.")